In [ ]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import benchutils as bu

plt.style.use('bmh')

# the interpreter running this notebook, so workers land in the same env
PYTHON_PATH = sys.executable
WORKER_SCRIPT = 'tmp/symscan_ncpu_worker.py'
N_SEQUENCE = 1_000_000
MAX_DISTANCE = 2
DISTANCE_TYPE = 'levenshtein'
N_REPS = 1

# Sweep 1..N threads over whatever cores this process was pinned to by
# bench_pinned.sh (one logical CPU per physical core, one core class).
MAX_NCPU = bu.available_cpus()
ncpus = np.arange(1, MAX_NCPU + 1, 1)
print(f'scaling over {MAX_NCPU} cores: {bu.affinity_list()}')

In [ ]:
bu.describe_env()

In [ ]:
!mkdir -p tmp

In [ ]:
%%writefile tmp/symscan_ncpu_worker.py
import sys
import time
import pandas as pd

import symscan


def main():
    n_sequence, max_distance, distance_type = sys.argv[1:5]
    n_sequence = int(n_sequence)
    max_distance = int(max_distance)

    N_FILES=1
    seqs = []
    for i in range(1,N_FILES+1):
        seqs += pd.read_csv(f'../data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()
    seqs = seqs[:n_sequence]

    t0 = time.perf_counter()
    symscan.get_neighbors_within(seqs, max_distance=max_distance, distance_type=distance_type)
    print(time.perf_counter() - t0)


if __name__ == '__main__':
    main()

In [ ]:
def measure_runtime_seconds(n_cpu, n_sequence=N_SEQUENCE, max_distance=MAX_DISTANCE, distance_type=DISTANCE_TYPE):
    cmd = [PYTHON_PATH, WORKER_SCRIPT, str(n_sequence), str(max_distance), distance_type]
    env = os.environ | {'RAYON_NUM_THREADS': str(n_cpu)}
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return float(result.stdout)


In [ ]:
rows = []
for rep in range(N_REPS):
    for ncpu in ncpus:
        runtime_s = measure_runtime_seconds(ncpu)
        rows.append({'algorithm': 'symscan', 'n_cpu': int(ncpu), 'n_sequence': N_SEQUENCE,
                      'distance': MAX_DISTANCE, 'measure': DISTANCE_TYPE,
                      'runtime_s': runtime_s})
        print(ncpu, rep, runtime_s)

ncpu_df = pd.DataFrame(rows)
ncpu_df.to_csv('../data/symscan_ncpu_benchmark.csv')